In [665]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [666]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import datacompy
import pytz
import gspread

from google.cloud import storage



In [667]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.PRODUCTION` 
WHERE CATEGORY_EVENT='Jump' AND RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ'  AND RESULT!='FOUL' AND RESULT IS NOT NULL

"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)




Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [668]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,23/02/2003,SGP,0.6,Jump,Male,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,14/10/2000,SGP,0.1,Jump,Female,3rd Central Asian Athletics Champs,2026-08-23 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,14/10/2000,SGP,0.8,Jump,Female,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,04/12/2008,SGP,,Jump,Male,Pesta Sukan Age-Category Champs,2026-07-29 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,21/04/2001,SGP,,Jump,Male,Pesta Sukan Age-Category Champs,2026-07-26 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,29/01/1965,,,Jump,Female,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00
23262,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,27/05/2015,,,Jump,Male,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00
23263,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,05/12/2015,,,Jump,Male,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00
23264,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,03/04/2017,,,Jump,Male,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00


In [669]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


competitors.to_csv('database_jumps.csv', sep=',', encoding='utf-8-sig', index=False)

In [670]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,23/02/2003,SGP,0.6,Jump,Male,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,14/10/2000,SGP,0.1,Jump,Female,3rd Central Asian Athletics Champs,2026-08-23 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,14/10/2000,SGP,0.8,Jump,Female,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,04/12/2008,SGP,,Jump,Male,Pesta Sukan Age-Category Champs,2026-07-29 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,21/04/2001,SGP,,Jump,Male,Pesta Sukan Age-Category Champs,2026-07-26 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,29/01/1965,,,Jump,Female,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00
23262,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,27/05/2015,,,Jump,Male,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00
23263,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,05/12/2015,,,Jump,Male,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00
23264,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,03/04/2017,,,Jump,Male,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00


In [671]:
# DATE column to contain timezone - tz aware mode

competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=True)


In [672]:
# datetime to contain UTC (timezone)

competitors['NOW'] = datetime.datetime.now()

timezone = pytz.timezone('UTC')

competitors['NOW'] = datetime.datetime.now().replace(tzinfo=timezone)

In [673]:
# Calculate number of days from today to event date

#competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=False)

competitors['delta_time'] = competitors['NOW'] - competitors['DATE']


#competitors['delta_time'] = datetime.datetime.now() - competitors['DATE']


competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['DATE'].dt.month

# Make sure date conversion is is valid for all rows

assert not competitors['delta_time'].isna().any()

In [674]:
'''
# Choose date range for report

competitors['DATE']=competitors['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64


start = datetime.datetime(2025, 1, 1)
end = datetime.datetime(2026, 8, 25)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = (competitors['DATE'] >= start_date) & (competitors['DATE'] <= end_date)
athletes_selected = competitors.loc[mask].copy()

'''

"\n# Choose date range for report\n\ncompetitors['DATE']=competitors['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64\n\n\nstart = datetime.datetime(2025, 1, 1)\nend = datetime.datetime(2026, 8, 25)\n\nstart_date = np.datetime64(start)\nend_date = np.datetime64(end)\n\n\nmask = (competitors['DATE'] >= start_date) & (competitors['DATE'] <= end_date)\nathletes_selected = competitors.loc[mask].copy()\n\n"

In [675]:
# ============================================================
# Choose calendar-date range for report
# Time-of-day is deliberately ignored for eligibility.
# ============================================================

start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2026, 8, 30)

# Keep original DATE intact, but derive calendar date for filtering
competitors['DATE_CALENDAR'] = (
    pd.to_datetime(
        competitors['DATE'],
        errors='coerce'
    )
    .dt.date
)

mask = (
    competitors['DATE_CALENDAR'].notna()
    & (competitors['DATE_CALENDAR'] >= start_date)
    & (competitors['DATE_CALENDAR'] <= end_date)
)

athletes_selected = competitors.loc[mask].copy()

In [676]:
end_date - start_date

datetime.timedelta(days=606)

In [677]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


athletes_selected.to_csv('athletes_selected_jumps_only.csv', encoding='utf-8')

In [678]:
athletes_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DATE_CALENDAR
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-23 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,7 days 17:30:05.372581,7,8,2026-08-23
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,Pesta Sukan Age-Category Champs,2026-07-29 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,32 days 17:30:05.372581,32,7,2026-07-29
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,Pesta Sukan Age-Category Champs,2026-07-26 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,35 days 17:30:05.372581,35,7,2026-07-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23262,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23263,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23264,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07


In [679]:
# Choose 2024/25 only

athletes = athletes_selected

In [680]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DATE_CALENDAR
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-23 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,7 days 17:30:05.372581,7,8,2026-08-23
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,Pesta Sukan Age-Category Champs,2026-07-29 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,32 days 17:30:05.372581,32,7,2026-07-29
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,Pesta Sukan Age-Category Champs,2026-07-26 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,35 days 17:30:05.372581,35,7,2026-07-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23262,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23263,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23264,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07


In [681]:
# Clean text columns only — preserve numeric/date dtypes
text_cols = athletes.select_dtypes(
    include=['object', 'string']
).columns

for col in text_cols:
    athletes[col] = (
        athletes[col]
        .fillna('')
        .astype(str)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'[\x00-\x1f\x7f-\x9f]', '', regex=True)
        .str.replace('\r', ' ', regex=False)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
    )

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.strip()
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

In [682]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DATE_CALENDAR
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-23 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,7 days 17:30:05.372581,7,8,2026-08-23
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,Pesta Sukan Age-Category Champs,2026-07-29 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,32 days 17:30:05.372581,32,7,2026-07-29
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,Pesta Sukan Age-Category Champs,2026-07-26 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,35 days 17:30:05.372581,35,7,2026-07-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23262,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23263,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23264,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07


In [683]:
# Read benchmarks file

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

benchmarks = pd.read_csv('Benchmarks.csv')


In [684]:
benchmarks

,GENDER,SQUAD,EVENT,BENCHMARK
0,Male,NATIONAL,Long Jump,7.20
1,Female,NATIONAL,Long Jump,5.70
2,Male,NATIONAL,Triple Jump,15.20
3,Female,NATIONAL,Triple Jump,12.00
4,Male,NATIONAL,High Jump,2.07
5,Female,NATIONAL,High Jump,1.65
6,Male,TRAINING,Long Jump,6.80
7,Female,TRAINING,Long Jump,5.30
8,Male,TRAINING,Triple Jump,14.30
9,Female,TRAINING,Triple Jump,11.30


In [685]:
def convert_time_refactored(i, string, metric):
    import datetime  # if not already imported above
    
    l = ['discus', 'throw', 'jump', 'vault', 'shot']
    sprint_events = ['100m', '200m', '400m']
    
    string = str(string).lower()
    metric_str = str(metric)
    output = ''
    
    try:
        # Skip marks with illegal wind speeds
        #if isinstance(metric_str, str) and 'w' in metric_str.lower():
        #    return ''
        
        ## Field events (distances)
      #  if any(s in string for s in l):
      #      metric_clean = metric_str.replace('m', '').replace('GR', '')
      #      return round(float(metric_clean), 2)

        if any(s in string for s in l):
            metric_clean = metric_str.replace('GR', '').strip()
        
        # Remove trailing unit/wind notation while retaining
        # the numeric jump mark.
        # Wind does not affect eligibility for Jumps Selection.
            metric_clean = re.sub(
                r'(?i)[mw]$',
                '',
                metric_clean
            ).strip()
        
            return round(float(metric_clean), 2)

        
        if string == '':
            return ''
        
        count_colon = metric_str.count(':')
        count_dot = metric_str.count('.')
        
        # Simple time as float (no colon)
        if count_colon == 0:
            return round(float(metric_str), 2)
        
        # Sprint events
        if any(sprint in string for sprint in sprint_events):
            if count_colon == 1 and count_dot == 1:
                parts = metric_str.split(':')
                if len(parts) == 2:
                    first_part, second_part = parts
                    if first_part == '00':
                        return float(second_part)
                    else:
                        return float(int(first_part) * 60 + float(second_part))

        # ---------- NEW: marathon “MM:SS.ss” fixer ----------
        # Marathon events sometimes wrongly stored as MM:SS.ss instead of HH:MM:SS.ss
        # e.g. "2:27.33" meaning 2h 27m 33s, but delivered as "2:27.33"
        if ('marathon' in string or '20 km racewalk' in string or '50 km racewalk' in string):
            if count_colon == 1 and count_dot == 1:
                # Interpret "MM:SS.ss" as "HH:MM:SS.ss" with HH = floor(MM / 60) or from context.
                # If your data encodes marathon as "2:27.33" (2h27m33s), treat first part as hours:
                h_str, ms_str = metric_str.split(':')
                h = int(h_str)               # hours
                m = int(ms_str.split('.')[0])  # minutes (before dot)
                s = float('0.' + ms_str.split('.')[1]) * 60  # fractional seconds
                return float(h * 3600 + m * 60 + s)
        # ---------------------------------------------------

        # Two colons
        if count_colon == 2:
            if count_dot == 0:
                h, m, s = metric_str.split(':')
                return float(int(h) * 3600 + int(m) * 60 + float(s))

            if ('10,000m' in string or '5000m' in string or '1500m' in string):
                if len(metric_str) == 7:  # X:XX:XX (1500m special case)
                    idx = 4
                    metric_mod = '0' + metric_str[:idx] + '.' + metric_str[idx+1:]
                else:
                    idx = 5
                    metric_mod = metric_str[:idx] + '.' + metric_str[idx+1:]
                m, s = metric_mod.split(':')[-2:]
                return float((int(m) * 60) + float(s))

            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # datetime
        if isinstance(metric, (datetime.time, datetime.datetime)):
            t = str(metric)
            h, m, s = t.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # MM:SS.sss
        if count_colon == 1 and count_dot >= 1:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + float(s))
        
        # HH.MM.SS or similar
        if count_colon == 1 and count_dot == 2:
            metric_mod = metric_str.replace('.', ':', 1)
            h, m, s = metric_mod.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # HH:MM:SS (no dots)
        if count_colon == 2 and count_dot == 0:
            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # MM:SS (no dots)
        if count_colon == 1 and count_dot == 0:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + int(s))
            
    except Exception:
        return ''
    
    return output


In [686]:
def process_benchmarks(df):
    
    for i in range(len(df)):

        rowIndex = df.index[i]

        input_string=df.iloc[rowIndex,0]
    
        metric=df.iloc[rowIndex,3]
    
        if metric==None:
        
            continue
        
        out = convert_time_refactored(i, input_string, metric)
        
        print(rowIndex, input_string, out)

    
        df.loc[rowIndex, 'Metric'] = out
    
    return df

In [687]:
process_benchmarks(benchmarks)

0 Male 7.2
1 Female 5.7
2 Male 15.2
3 Female 12.0
4 Male 2.07
5 Female 1.65
6 Male 6.8
7 Female 5.3
8 Male 14.3
9 Female 11.3
10 Male 1.95
11 Female 1.57


,GENDER,SQUAD,EVENT,BENCHMARK,Metric
0,Male,NATIONAL,Long Jump,7.20,7.20
1,Female,NATIONAL,Long Jump,5.70,5.70
2,Male,NATIONAL,Triple Jump,15.20,15.20
3,Female,NATIONAL,Triple Jump,12.00,12.00
4,Male,NATIONAL,High Jump,2.07,2.07
5,Female,NATIONAL,High Jump,1.65,1.65
6,Male,TRAINING,Long Jump,6.80,6.80
7,Female,TRAINING,Long Jump,5.30,5.30
8,Male,TRAINING,Triple Jump,14.30,14.30
9,Female,TRAINING,Triple Jump,11.30,11.30


In [688]:
mask = benchmarks['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)

benchmarks.loc[mask, '2%']=benchmarks['BENCHMARK']*0.98
benchmarks.loc[mask, '3.5%']=benchmarks['BENCHMARK']*0.965
benchmarks.loc[mask, '5%']=benchmarks['BENCHMARK']*0.95
benchmarks.loc[mask, '10%']=benchmarks['BENCHMARK']*0.90


benchmarks.loc[~mask, '2%']=benchmarks['BENCHMARK']*1.02
benchmarks.loc[~mask, '3.5%']=benchmarks['BENCHMARK']*1.035
benchmarks.loc[~mask, '5%']=benchmarks['BENCHMARK']*1.05
benchmarks.loc[~mask, '10%']=benchmarks['BENCHMARK']*1.10


In [689]:
# ============================================================
# CLEAN BENCHMARK DATA WITHOUT CONVERTING ALL COLUMNS TO STRING
# ============================================================

# Clean text columns only
text_cols = benchmarks.select_dtypes(
    include=['object', 'string']
).columns

for col in text_cols:
    benchmarks[col] = (
        benchmarks[col]
        .fillna('')
        .astype(str)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'[\x00-\x1f\x7f-\x9f]', '', regex=True)
        .str.replace('\r', ' ', regex=False)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
    )

# Ensure benchmark values are numeric for calculations
benchmarks['BENCHMARK'] = pd.to_numeric(
    benchmarks['BENCHMARK'],
    errors='coerce'
)

In [690]:
benchmarks

,GENDER,SQUAD,EVENT,BENCHMARK,Metric,2%,3.5%,5%,10%
0,Male,NATIONAL,Long Jump,7.20,7.20,7.0560,6.94800,6.8400,6.480
1,Female,NATIONAL,Long Jump,5.70,5.70,5.5860,5.50050,5.4150,5.130
2,Male,NATIONAL,Triple Jump,15.20,15.20,14.8960,14.66800,14.4400,13.680
3,Female,NATIONAL,Triple Jump,12.00,12.00,11.7600,11.58000,11.4000,10.800
4,Male,NATIONAL,High Jump,2.07,2.07,2.0286,1.99755,1.9665,1.863
5,Female,NATIONAL,High Jump,1.65,1.65,1.6170,1.59225,1.5675,1.485
6,Male,TRAINING,Long Jump,6.80,6.80,6.6640,6.56200,6.4600,6.120
7,Female,TRAINING,Long Jump,5.30,5.30,5.1940,5.11450,5.0350,4.770
8,Male,TRAINING,Triple Jump,14.30,14.30,14.0140,13.79950,13.5850,12.870
9,Female,TRAINING,Triple Jump,11.30,11.30,11.0740,10.90450,10.7350,10.170


In [691]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DATE_CALENDAR
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-23 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,7 days 17:30:05.372581,7,8,2026-08-23
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,3rd Central Asian Athletics Champs,2026-08-24 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,6 days 17:30:05.372581,6,8,2026-08-24
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,Pesta Sukan Age-Category Champs,2026-07-29 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,32 days 17:30:05.372581,32,7,2026-07-29
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,Pesta Sukan Age-Category Champs,2026-07-26 00:00:00+00:00,2026,International,2026-08-30 09:31:00+00:00,2026-08-30 17:30:05.372581+00:00,35 days 17:30:05.372581,35,7,2026-07-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23262,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23263,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07
23264,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,SMTFA International Masters Track - Field Cham...,2026-06-07 00:00:00+00:00,2026,Local,2026-08-17 09:02:00+00:00,2026-08-30 17:30:05.372581+00:00,84 days 17:30:05.372581,84,6,2026-06-07


In [692]:
'''
# Select NATIONAL or TRAINING benchmarks

benchmarks = benchmarks[benchmarks['SQUAD']=='TRAINING']

'''

"\n# Select NATIONAL or TRAINING benchmarks\n\nbenchmarks = benchmarks[benchmarks['SQUAD']=='TRAINING']\n\n"

In [693]:
# ============================================================
# REFERENCE BENCHMARK SET FOR BEST-RESULT SELECTION
# ============================================================

# Keep the full benchmarks dataframe intact because we will
# subsequently calculate both TRAINING and NATIONAL selections.

reference_benchmarks = benchmarks.loc[
    benchmarks['SQUAD']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
    .eq('NATIONAL')
].copy()

In [694]:
benchmarks

,GENDER,SQUAD,EVENT,BENCHMARK,Metric,2%,3.5%,5%,10%
0,Male,NATIONAL,Long Jump,7.20,7.20,7.0560,6.94800,6.8400,6.480
1,Female,NATIONAL,Long Jump,5.70,5.70,5.5860,5.50050,5.4150,5.130
2,Male,NATIONAL,Triple Jump,15.20,15.20,14.8960,14.66800,14.4400,13.680
3,Female,NATIONAL,Triple Jump,12.00,12.00,11.7600,11.58000,11.4000,10.800
4,Male,NATIONAL,High Jump,2.07,2.07,2.0286,1.99755,1.9665,1.863
5,Female,NATIONAL,High Jump,1.65,1.65,1.6170,1.59225,1.5675,1.485
6,Male,TRAINING,Long Jump,6.80,6.80,6.6640,6.56200,6.4600,6.120
7,Female,TRAINING,Long Jump,5.30,5.30,5.1940,5.11450,5.0350,4.770
8,Male,TRAINING,Triple Jump,14.30,14.30,14.0140,13.79950,13.5850,12.870
9,Female,TRAINING,Triple Jump,11.30,11.30,11.0740,10.90450,10.7350,10.170


In [695]:
'''
# Merge benchmarks onto athletes on MAPPED_EVENT and GENDER

df = pd.merge(
    left=athletes, 
    right=benchmarks,
    how='left',
    left_on=['EVENT', 'GENDER'],
    right_on=['EVENT', 'GENDER'],
)
'''

"\n# Merge benchmarks onto athletes on MAPPED_EVENT and GENDER\n\ndf = pd.merge(\n    left=athletes, \n    right=benchmarks,\n    how='left',\n    left_on=['EVENT', 'GENDER'],\n    right_on=['EVENT', 'GENDER'],\n)\n"

In [696]:
df = pd.merge(
    left=athletes,
    right=reference_benchmarks,
    how='left',
    on=['EVENT', 'GENDER'],
)

In [697]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,delta_time_conv,event_month,DATE_CALENDAR,SQUAD,BENCHMARK,Metric,2%,3.5%,5%,10%
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,6,8,2026-08-24,NATIONAL,15.20,15.20,14.8960,14.66800,14.4400,13.680
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,7,8,2026-08-23,NATIONAL,5.70,5.70,5.5860,5.50050,5.4150,5.130
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,6,8,2026-08-24,NATIONAL,12.00,12.00,11.7600,11.58000,11.4000,10.800
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,32,7,2026-07-29,NATIONAL,2.07,2.07,2.0286,1.99755,1.9665,1.863
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,35,7,2026-07-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5250,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,84,6,2026-06-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5251,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,84,6,2026-06-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5252,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,84,6,2026-06-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5253,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,84,6,2026-06-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [698]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


df.to_csv('jumps_postmap_30aug2026.csv', sep=',', encoding='utf-8-sig', index=False)


In [699]:
# Convert results and seed into seconds format for mapped events only (vectorised version)
# Clean text columns only — preserve numeric/date dtypes

text_cols = df.select_dtypes(
    include=['object', 'string']
).columns

for col in text_cols:
    df[col] = (
        df[col]
        .fillna('')
        .astype(str)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'[\x00-\x1f\x7f-\x9f]', '', regex=True)
        .str.replace('\r', ' ', regex=False)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
    )
# Define a filter for rows with convertible results
invalid_results = {'—', 'None', 'DQ', 'SCR', 'FS', 'DNQ', 'DNS', 'NH', 'NM', 'FOUL', 'DNF', 'SR'}

# Apply conversion vectorized using apply, skipping invalid values
def convert_for_row(row):
    if row['RESULT'] in invalid_results:
        return ''
    return convert_time_refactored(row.name, row['EVENT'], row['RESULT'])

df['RESULT_CONV'] = df.apply(convert_for_row, axis=1)


In [700]:
# Change to numeric

df[['2%', '3.5%', '5%', '10%', 'RESULT_CONV', 'BENCHMARK']] = df[['2%', '3.5%', '5%', '10%', 'RESULT_CONV', 'BENCHMARK']].apply(pd.to_numeric, errors='coerce')

In [701]:
mask = df['CATEGORY_EVENT'].str.lower().str.contains(r'jump|throw|decathlon|heptathlon', na=True)

df.loc[mask, 'Delta2'] = df['RESULT_CONV']-df['2%']
df.loc[mask, 'Delta3.5'] = df['RESULT_CONV']-df['3.5%']
df.loc[mask, 'Delta5'] = df['RESULT_CONV']-df['5%']
df.loc[mask, 'Delta10'] = df['RESULT_CONV']-df['10%']
df.loc[mask, 'Delta_Benchmark'] = df['RESULT_CONV']-df['BENCHMARK']

df.loc[~mask, 'Delta2'] =  df['2%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta3.5'] = df['3.5%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta5'] = df['5%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta10'] = df['10%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta_Benchmark'] = df['BENCHMARK'] - df['RESULT_CONV']



In [702]:
# Performance metric to filter out athletes

df['PERF_SCALAR']=df['Delta5']/df['BENCHMARK']*100

In [703]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


df.to_csv('jumps_postmap_benchmarked_30aug2026.csv', sep=',', encoding='utf-8-sig', index=False)


In [704]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,14.66800,14.4400,13.680,15.19,0.2940,0.52200,0.7500,1.510,-0.01,4.934211
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,5.50050,5.4150,5.130,6.08,0.4940,0.57950,0.6650,0.950,0.38,11.666667
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,11.58000,11.4000,10.800,12.68,0.9200,1.10000,1.2800,1.880,0.68,10.666667
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,1.99755,1.9665,1.863,2.04,0.0114,0.04245,0.0735,0.177,-0.03,3.550725
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,NaN,NaN,NaN,4.88,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5250,"Foo, Belinda",1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.90,NaN,NaN,NaN,NaN,NaN,NaN
5251,"Poh, Tristan",1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.05,NaN,NaN,NaN,NaN,NaN,NaN
5252,"Tan, Robin Yu Le",1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.65,NaN,NaN,NaN,NaN,NaN,NaN
5253,"Poh, Evan",1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.45,NaN,NaN,NaN,NaN,NaN,NaN


In [705]:

# Read a variation name list and corrections from CSVs
'''
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/')

names = pd.read_csv("name_variations.csv")

for index, row in names.iterrows():
        
    print(names.VARIATION, names.NAME)
    df['NAME'] = df['NAME'].replace(regex=rf"{row['VARIATION']}", value=f"{row['NAME']}")
'''

'\nos.chdir(\'/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/\')\n\nnames = pd.read_csv("name_variations.csv")\n\nfor index, row in names.iterrows():\n        \n    print(names.VARIATION, names.NAME)\n    df[\'NAME\'] = df[\'NAME\'].replace(regex=rf"{row[\'VARIATION\']}", value=f"{row[\'NAME\']}")\n'

In [706]:
def clean_base_name(x):
    """
    Basic cleanup:
    - remove NBSP
    - remove control characters
    - replace line breaks with spaces
    - collapse repeated spaces
    - strip surrounding spaces
    """
    if pd.isna(x):
        return ""

    x = str(x)
    x = x.replace("\xa0", " ")
    x = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x)

    return x.strip()


def strip_punctuation(x):
    """
    Remove Unicode punctuation such as:
    commas, full stops, hyphens, apostrophes and slashes.
    """
    return "".join(
        ch
        for ch in x
        if not unicodedata.category(ch).startswith("P")
    )


def _octc_name_match_key(x):
    """
    Create a standard name-matching key.

    This function:
    - performs basic text cleanup
    - explicitly removes regex anchors ^ and $
    - removes regex escape characters
    - removes punctuation
    - removes spaces
    - compares names without regard to case

    Examples:
    '^Chen Xiang Ang$' -> 'chenxiangang'
    'Chen Xiang Ang'   -> 'chenxiangang'

    '^Tan Tate$'       -> 'tantate'
    'Tan Tate'         -> 'tantate'
    """
    x = clean_base_name(x)

    # Explicitly remove regex characters that are not classified
    # as ordinary Unicode punctuation.
    x = (
        x
        .replace("^", "")
        .replace("$", "")
        .replace("\\", "")
    )

    x = strip_punctuation(x)
    x = re.sub(r"\s+", "", x)

    return x.casefold()


def clean_replacement_name(x):
    """
    Clean the canonical replacement name while retaining normal
    spaces between words.
    """
    x = clean_base_name(x)
    x = strip_punctuation(x)
    x = re.sub(r"\s+", " ", x)

    return x.strip().casefold()

def name_token_signature(x):
    """
    Create an order-insensitive token signature for a name.

    Used only as a conservative fallback after exact
    variation-key matching.

    Example:
        'Shyen Joshua Lee'  -> 'joshua|lee|shyen'
        'Joshua Shyen Lee'  -> 'joshua|lee|shyen'
        'Lee Joshua Shyen'  -> 'joshua|lee|shyen'
    """

    x = clean_base_name(x)
    x = strip_punctuation(x)

    tokens = re.findall(
        r'[a-z0-9]+',
        x.casefold()
    )

    return '|'.join(sorted(tokens))

In [707]:
# ============================================================
# STANDARDISE ATHLETE NAMES USING GCS NAME VARIATIONS
# ============================================================

# Create exact matching keys and conservative token-order signatures
# for athlete names in the results dataframe.

df["NAME_KEY"] = df["NAME"].apply(
    _octc_name_match_key
)

df["NAME_TOKEN_SIGNATURE"] = df["NAME"].apply(
    name_token_signature
)


# ============================================================
# READ NAME VARIATIONS
# ============================================================

file_path = "gs://name_variations/name_variations.csv"

names = pd.read_csv(
    file_path,
    sep=",",
    storage_options={
        "token": (
            "/Users/veesheenyuen/Desktop/DataScience/Keys/"
            "saa-analytics-7c8937b70609.json"
        )
    },
)


# Remove hidden spaces from column headings
names.columns = (
    names.columns
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)


# ============================================================
# CREATE MATCHING KEYS
# ============================================================

names["VARIATION_KEY"] = names["VARIATION"].apply(
    _octc_name_match_key
)

names["NAME_CLEAN"] = names["NAME"].apply(
    clean_replacement_name
)

names["TOKEN_SIGNATURE"] = names["VARIATION"].apply(
    name_token_signature
)


# Exclude completely blank variation keys
names = names[
    names["VARIATION_KEY"]
    .fillna("")
    .astype(str)
    .str.strip()
    != ""
].copy()


# ============================================================
# EXACT VARIATION-KEY MAP
# ============================================================

# Exact matching remains the primary name-matching method.

name_map = (
    names
    .drop_duplicates(
        subset=["VARIATION_KEY"],
        keep="last",
    )
    .set_index("VARIATION_KEY")["NAME_CLEAN"]
    .to_dict()
)


# ============================================================
# UNAMBIGUOUS TOKEN-ORDER FALLBACK
# ============================================================

# Group all variations having the same collection of name tokens.
#
# Example:
#
#   Shyen Joshua Lee
#   Joshua Shyen Lee
#   Lee Joshua Shyen
#
# all generate:
#
#   joshua|lee|shyen
#
# The fallback is used ONLY if every matching variation resolves
# to exactly one canonical athlete.

signature_candidates = (
    names.loc[
        names["TOKEN_SIGNATURE"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    ]
    .groupby(
        "TOKEN_SIGNATURE"
    )["NAME_CLEAN"]
    .agg(
        lambda values: sorted({
            str(value).strip()
            for value in values
            if str(value).strip()
        })
    )
)


# Do not automatically resolve ambiguous signatures.

ambiguous_token_signatures = signature_candidates.loc[
    signature_candidates.apply(len) > 1
]


if not ambiguous_token_signatures.empty:

    print(
        "Ambiguous token-order name signatures excluded from fallback:",
        len(ambiguous_token_signatures)
    )


token_signature_map = {

    signature: canonical_names[0]

    for signature, canonical_names
    in signature_candidates.items()

    if len(canonical_names) == 1
}


# ============================================================
# APPLY NAME STANDARDISATION
# ============================================================

# Exact variation match first.

exact_match = (
    df["NAME_KEY"]
    .map(name_map)
)


# Token-order fallback second.

token_order_fallback = (
    df["NAME_TOKEN_SIGNATURE"]
    .map(token_signature_map)
)


# Exact variation mapping takes precedence.
#
# If exact matching fails, use the token-order fallback.
# If both fail, retain the cleaned original name.

df["NAME"] = (
    exact_match
    .fillna(token_order_fallback)
    .fillna(
        df["NAME"].apply(
            clean_replacement_name
        )
    )
)


df["NAME"] = df["NAME"].str.title()


# ============================================================
# TOKEN-ORDER FALLBACK AUDIT
# ============================================================

token_fallback_audit = df.loc[

    exact_match.isna()
    & token_order_fallback.notna(),

    [
        "NAME",
        "NAME_KEY",
        "NAME_TOKEN_SIGNATURE",
    ]

].copy()


if not token_fallback_audit.empty:

    print(
        "Names resolved by unambiguous token-order fallback:",
        len(token_fallback_audit)
    )

    display(
        token_fallback_audit
        .drop_duplicates()
    )


# Remove temporary keys

df = df.drop(
    columns=[
        "NAME_KEY",
        "NAME_TOKEN_SIGNATURE",
    ]
)

Ambiguous token-order name signatures excluded from fallback: 2
Names resolved by unambiguous token-order fallback: 37


,NAME,NAME_KEY,NAME_TOKEN_SIGNATURE
7,Ong Yuxi Ashlee,ongyuxiashlee,ashlee|ong|yuxi
106,Tan Elijah,tanelijah,elijah|tan
524,Lau Jia Hern,jiahernlau,hern|jia|lau
531,Tan Shou Yi Rei,reitan,rei|tan
544,Goh Yen Young Amelia,ameliagoh,amelia|goh
613,Goh Yen Young Amelia,yenyoungameliagoh,amelia|goh|yen|young
1107,Zhao Daniel,zhaodaniel,daniel|zhao
1548,Feng Han Lin,fenghanlin,feng|han|lin
2488,Yen Ming Zhen,yenmingzhenwinter,ming|winter|yen|zhen
3320,Ong Yuxi Ashlee,ashleeong,ashlee|ong


In [708]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,14.66800,14.4400,13.680,15.19,0.2940,0.52200,0.7500,1.510,-0.01,4.934211
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,5.50050,5.4150,5.130,6.08,0.4940,0.57950,0.6650,0.950,0.38,11.666667
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,11.58000,11.4000,10.800,12.68,0.9200,1.10000,1.2800,1.880,0.68,10.666667
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,1.99755,1.9665,1.863,2.04,0.0114,0.04245,0.0735,0.177,-0.03,3.550725
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,NaN,NaN,NaN,4.88,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5250,Foo Belinda,1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.90,NaN,NaN,NaN,NaN,NaN,NaN
5251,Poh Tristan,1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.05,NaN,NaN,NaN,NaN,NaN,NaN
5252,Tan Robin Yu Le,1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.65,NaN,NaN,NaN,NaN,NaN,NaN
5253,Poh Evan,1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,NaN,NaN,NaN,1.45,NaN,NaN,NaN,NaN,NaN,NaN


In [709]:
# Choose only Singaporeans

#allowed_nationalities = ['SGP', 'SIN', 'NONE', '']

#df_select = df[
#    df['NATIONALITY']
#        .fillna('')
#        .astype(str)
#        .str.strip()
#        .str.upper()
#        .isin(allowed_nationalities)
#].copy()


# Choose only Singaporeans

allowed_nationalities = ['SGP', 'SIN', 'NONE', '']

nationality_mask = (
    df['NATIONALITY']
        .fillna('')
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(allowed_nationalities)
)

df_select = df.loc[nationality_mask].copy()

In [710]:
# ============================================================
# JUMPS WIND INFORMATION
# ============================================================

# Wind is informational only for Jumps Selection.
# It does not affect result eligibility for Long Jump,
# Triple Jump or High Jump.

wind_text = (
    df_select['WIND']
    .fillna('')
    .astype(str)
    .str.strip()
)

df_select['WIND_NUM'] = pd.to_numeric(
    wind_text.str.replace('+', '', regex=False),
    errors='coerce'
)

In [711]:
df_select

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,WIND_NUM
0,Lee Jing Yi Gabriel,15.19,SGP,,3,,Triple Jump,,,G897C03,...,14.4400,13.680,15.19,0.2940,0.52200,0.7500,1.510,-0.01,4.934211,0.6
1,Rozario Tia Louise,6.08,SGP,,3,,Long Jump,,,T704F00,...,5.4150,5.130,6.08,0.4940,0.57950,0.6650,0.950,0.38,11.666667,0.1
2,Rozario Tia Louise,12.68,SGP,,2,,Triple Jump,,,T704F00,...,11.4000,10.800,12.68,0.9200,1.10000,1.2800,1.880,0.68,10.666667,0.8
3,Tan Shou Yi Rei,2.04,SGP,,1,Senior,High Jump,,,R373G08,...,1.9665,1.863,2.04,0.0114,0.04245,0.0735,0.177,-0.03,3.550725,NaN
4,Low Jun Yu,4.88,SGP,,1,Senior,Pole Vault,,,J029E01,...,NaN,NaN,4.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5250,Foo Belinda,1.90m,Singapore Masters Track and Fi,61.0,1,Masters,Pole Vault,0.0,,,...,NaN,NaN,1.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5251,Poh Tristan,1.05m,DYPV,11.0,1,U13,Pole Vault,0.0,,,...,NaN,NaN,1.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5252,Tan Robin Yu Le,1.65m,DYPV,10.0,1,U13,Pole Vault,0.0,,,...,NaN,NaN,1.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5253,Poh Evan,1.45m,DYPV,9.0,2,U13,Pole Vault,0.0,,,...,NaN,NaN,1.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [712]:
df_PV = df_select[(df_select['EVENT']=='Pole Vault') & (df_select['GENDER']=='Female')]

In [713]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

df_PV.to_csv('check.csv', encoding='utf-8')

In [714]:
df2 = df_select.copy()

df2['PERF_SCALAR'] = pd.to_numeric(
    df2['PERF_SCALAR'],
    errors='coerce'
)

finite_mask = np.isfinite(
    df2['PERF_SCALAR']
)

# All valid numeric jump performances are eligible.
# Wind does not affect Jumps Selection.
df_best_candidates = df2.loc[
    finite_mask
].copy()

top_performers_clean = (
    df_best_candidates
    .sort_values(
        ['EVENT', 'GENDER', 'NAME', 'PERF_SCALAR'],
        ascending=[True, True, True, False]
    )
    .drop_duplicates(
        subset=['EVENT', 'GENDER', 'NAME'],
        keep='first'
    )
    .reset_index(drop=True)
)

In [715]:
top_performers_clean

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,WIND_NUM
0,Aleksandra Te Rai,1.20,SSP,,19.0,C,High Jump,,,,...,1.5675,1.485,1.20,-0.417,-0.39225,-0.3675,-0.285,-0.45,-22.272727,NaN
1,Amanda Sim Jia Hui,1.20,DHS,,18.0,C,High Jump,,,,...,1.5675,1.485,1.20,-0.417,-0.39225,-0.3675,-0.285,-0.45,-22.272727,NaN
2,Amelia Tiara,1.44m,Temasek Polytechnic,0.0,2,Open,High Jump,0.0,,,...,1.5675,1.485,1.44,-0.177,-0.15225,-0.1275,-0.045,-0.21,-7.727273,NaN
3,Ang Hui Yi,1.60m,Nanyang Polytechnic,18,1,U20,High Jump,0.0,,,...,1.5675,1.485,1.60,-0.017,0.00775,0.0325,0.115,-0.05,1.969697,NaN
4,Ang Yu Xia,1.43,SNG,,7.0,B,High Jump,,,,...,1.5675,1.485,1.43,-0.187,-0.16225,-0.1375,-0.055,-0.22,-8.333333,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2352,Yuan Yinze,13.31,EJC,,3.0,A,Triple Jump,,,,...,14.4400,13.680,13.31,-1.586,-1.35800,-1.1300,-0.370,-1.89,-7.434211,NaN
2353,Zachary Tang,11.30,St. Joseph's Institution -,,2,,Triple Jump,,nan,,...,14.4400,13.680,11.30,-3.596,-3.36800,-3.1400,-2.380,-3.90,-20.657895,2.4
2354,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,,...,14.4400,13.680,11.98,-2.916,-2.68800,-2.4600,-1.700,-3.22,-16.184211,1.2
2355,Zheng Justin De,10.29,NJC,,12.0,C,Triple Jump,,,,...,14.4400,13.680,10.29,-4.606,-4.37800,-4.1500,-3.390,-4.91,-27.302632,NaN


In [716]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

top_performers_clean.to_csv('jumps_top_performers_30aug2026.csv', encoding='utf-8')

In [717]:
# Choose best performance for each event

#tiered_performers = top_performers_clean.sort_values(['GENDER', 'MAPPED_EVENT', 'PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)

tiered_performers = top_performers_clean


In [718]:
tiered_performers

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,WIND_NUM
0,Aleksandra Te Rai,1.20,SSP,,19.0,C,High Jump,,,,...,1.5675,1.485,1.20,-0.417,-0.39225,-0.3675,-0.285,-0.45,-22.272727,NaN
1,Amanda Sim Jia Hui,1.20,DHS,,18.0,C,High Jump,,,,...,1.5675,1.485,1.20,-0.417,-0.39225,-0.3675,-0.285,-0.45,-22.272727,NaN
2,Amelia Tiara,1.44m,Temasek Polytechnic,0.0,2,Open,High Jump,0.0,,,...,1.5675,1.485,1.44,-0.177,-0.15225,-0.1275,-0.045,-0.21,-7.727273,NaN
3,Ang Hui Yi,1.60m,Nanyang Polytechnic,18,1,U20,High Jump,0.0,,,...,1.5675,1.485,1.60,-0.017,0.00775,0.0325,0.115,-0.05,1.969697,NaN
4,Ang Yu Xia,1.43,SNG,,7.0,B,High Jump,,,,...,1.5675,1.485,1.43,-0.187,-0.16225,-0.1375,-0.055,-0.22,-8.333333,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2352,Yuan Yinze,13.31,EJC,,3.0,A,Triple Jump,,,,...,14.4400,13.680,13.31,-1.586,-1.35800,-1.1300,-0.370,-1.89,-7.434211,NaN
2353,Zachary Tang,11.30,St. Joseph's Institution -,,2,,Triple Jump,,nan,,...,14.4400,13.680,11.30,-3.596,-3.36800,-3.1400,-2.380,-3.90,-20.657895,2.4
2354,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,,...,14.4400,13.680,11.98,-2.916,-2.68800,-2.4600,-1.700,-3.22,-16.184211,1.2
2355,Zheng Justin De,10.29,NJC,,12.0,C,Triple Jump,,,,...,14.4400,13.680,10.29,-4.606,-4.37800,-4.1500,-3.390,-4.91,-27.302632,NaN


In [719]:
'''
# Identify Tier 1/2/3 performers

#top_performers_clean['TIER'] = np.where((top_performers_clean['Delta_Benchmark']>=0), 'Tier 1',    
#                                np.where(((top_performers_clean['Delta_Benchmark']<0) & (top_performers_clean['Delta2']>=0)), 'Tier2',
#                                np.where(((top_performers_clean['Delta2']<0) & (top_performers_clean['Delta3.5']>=0)), 'Tier3', ' ')))


tiered_performers['TIER'] = np.where((tiered_performers['Delta_Benchmark']>=0), 'Tier 1',    
                                np.where(((tiered_performers['Delta_Benchmark']<0) & (tiered_performers['Delta2']>=0)), 'Tier 2',
                                np.where(((tiered_performers['Delta2']<0) & (tiered_performers['Delta3.5']>=0)), 'Tier 3',
                                np.where(((tiered_performers['Delta3.5']<0) & (tiered_performers['Delta5']>=0)), 'Tier 4',
                                np.where(((tiered_performers['Delta5']<0) & (tiered_performers['Delta10']>=0)), 'Tier 5', ' ')))))

'''

"\n# Identify Tier 1/2/3 performers\n\n#top_performers_clean['TIER'] = np.where((top_performers_clean['Delta_Benchmark']>=0), 'Tier 1',    \n#                                np.where(((top_performers_clean['Delta_Benchmark']<0) & (top_performers_clean['Delta2']>=0)), 'Tier2',\n#                                np.where(((top_performers_clean['Delta2']<0) & (top_performers_clean['Delta3.5']>=0)), 'Tier3', ' ')))\n\n\ntiered_performers['TIER'] = np.where((tiered_performers['Delta_Benchmark']>=0), 'Tier 1',    \n                                np.where(((tiered_performers['Delta_Benchmark']<0) & (tiered_performers['Delta2']>=0)), 'Tier 2',\n                                np.where(((tiered_performers['Delta2']<0) & (tiered_performers['Delta3.5']>=0)), 'Tier 3',\n                                np.where(((tiered_performers['Delta3.5']<0) & (tiered_performers['Delta5']>=0)), 'Tier 4',\n                                np.where(((tiered_performers['Delta5']<0) & (tiered_performers['Delta10']>=

In [720]:
'''
# Drop rows without a benchmark

final_df = tiered_performers[tiered_performers['BENCHMARK'].notna()].copy()
'''

"\n# Drop rows without a benchmark\n\nfinal_df = tiered_performers[tiered_performers['BENCHMARK'].notna()].copy()\n"

In [721]:
'''
final_df

'''

'\nfinal_df\n\n'

In [722]:
'''
final_tiered_selection = final_df[final_df['TIER']!=' ']
'''

"\nfinal_tiered_selection = final_df[final_df['TIER']!=' ']\n"

In [723]:
'''
final_tiered_selection.columns
'''

'\nfinal_tiered_selection.columns\n'

In [724]:
'''
final_tiered_selection = (
    final_tiered_selection.sort_values(
        ["EVENT", "GENDER", "PERF_SCALAR"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

'''

'\nfinal_tiered_selection = (\n    final_tiered_selection.sort_values(\n        ["EVENT", "GENDER", "PERF_SCALAR"],\n        ascending=[True, True, False]\n    )\n    .reset_index(drop=True)\n)\n\n'

In [725]:
# ============================================================
# SHARED JUMPS SPEX EXCLUSIONS BEFORE COMBINED SELECTION
# ============================================================

# Single source of truth used by:
#   1. combined final_tiered_selection
#   2. Programme Changes snapshots

JUMPS_SPEX_EXCLUDED_NAMES = [

    # spexPotential
    'Tia Louise Rozario',
    'Andrew George Medina',
    'Gabriel Lee Jing Yi',
    'Mark Lee Ren',
    'Reuben Rainer Lee Siong En',
    'Elizabeth Ann Tan Shee Ru',
    'Thiruben S/O Thana Rajan',

    # spexScholarship
    'Pereira Veronica Shanti',
    'Kampton Kam',
    'Ang Chen Xiang',
    'Quek Jun Jie Calvin',
    'Marc Brian Louis',
]


def jumps_spex_name_signature(value):
    """
    Order-insensitive signature used only
    for the explicit SPEX exclusion list.
    """

    tokens = re.findall(
        r'[a-z0-9]+',
        str(value).casefold()
    )

    return '|'.join(sorted(tokens))


def canonicalise_jumps_spex_name(value):
    """
    Canonicalise an exclusion name through the same
    athlete-name mapping used earlier in the notebook.
    """

    try:

        exact_key = _octc_name_match_key(value)

        mapped = name_map.get(
            exact_key
        )

        # If exact variation matching failed,
        # use the safe token-order fallback.
        if mapped is None:

            mapped = token_signature_map.get(
                name_token_signature(value),
                value
            )

        return clean_replacement_name(
            mapped
        ).title()

    except Exception:

        return str(
            value
        ).strip().title()


# Canonicalise all 12 explicit names.

canonical_jumps_spex_names = [

    canonicalise_jumps_spex_name(name)

    for name
    in JUMPS_SPEX_EXCLUDED_NAMES
]


# Build signatures from both:
# - the explicit list
# - their canonical forms

jumps_spex_excluded_signatures = {

    jumps_spex_name_signature(name)

    for name in (
        JUMPS_SPEX_EXCLUDED_NAMES
        + canonical_jumps_spex_names
    )

    if jumps_spex_name_signature(name)
}


# ============================================================
# REMOVE SPEX ATHLETES BEFORE CELL 61
# ============================================================

top_performers_clean['_SPEX_SIGNATURE'] = (

    top_performers_clean['NAME']
    .apply(
        jumps_spex_name_signature
    )

)


jumps_spex_exclusion_audit = (

    top_performers_clean.loc[

        top_performers_clean[
            '_SPEX_SIGNATURE'
        ].isin(
            jumps_spex_excluded_signatures
        )

    ]
    .copy()

)


top_performers_clean = (

    top_performers_clean.loc[

        ~top_performers_clean[
            '_SPEX_SIGNATURE'
        ].isin(
            jumps_spex_excluded_signatures
        )

    ]

    .drop(
        columns=[
            '_SPEX_SIGNATURE'
        ]
    )

    .reset_index(
        drop=True
    )

)


print(
    'SPEX athletes excluded before combined selection:',
    jumps_spex_exclusion_audit['NAME'].nunique()
)


if not jumps_spex_exclusion_audit.empty:

    display(

        jumps_spex_exclusion_audit[
            [
                'NAME',
                'GENDER',
                'EVENT',
            ]
        ]

        .drop_duplicates()

        .sort_values(
            [
                'NAME',
                'EVENT',
            ]
        )

    )

SPEX athletes excluded before combined selection: 4


,NAME,GENDER,EVENT
279,Kam Kampton,Male,High Jump
1460,Lee Jing Yi Gabriel,Male,Long Jump
2216,Lee Jing Yi Gabriel,Male,Triple Jump
311,Medina Andrew George,Male,High Jump
1557,Medina Andrew George,Male,Long Jump
2243,Medina Andrew George,Male,Triple Jump
894,Rozario Tia Louise,Female,Long Jump
2059,Rozario Tia Louise,Female,Triple Jump


In [726]:
# ============================================================
# COMBINED TRAINING + NATIONAL JUMPS SELECTION
# ============================================================

# top_performers_clean already contains one best performance
# per athlete + gender + event.
#
# We now evaluate that same best performance independently
# against BOTH Training and National benchmark structures.


# ============================================================
# 1. BUILD CLEAN BEST-RESULT BASE
# ============================================================

best_results_base = top_performers_clean.copy()

# Remove benchmark-specific/calculated columns inherited from
# the reference benchmark merge. They will be recalculated
# independently for Training and National.

benchmark_derived_cols = [
    'SQUAD',
    'BENCHMARK',
    'Metric',
    '2%',
    '3.5%',
    '5%',
    '10%',
    'Delta2',
    'Delta3.5',
    'Delta5',
    'Delta10',
    'Delta_Benchmark',
    'PERF_SCALAR',
    'TIER',
]

best_results_base = best_results_base.drop(
    columns=[
        col
        for col in benchmark_derived_cols
        if col in best_results_base.columns
    ],
    errors='ignore',
)


# ============================================================
# 2. PREPARE BOTH BENCHMARK SETS
# ============================================================

combined_benchmarks = benchmarks.copy()

combined_benchmarks['SQUAD'] = (
    combined_benchmarks['SQUAD']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
)

combined_benchmarks = combined_benchmarks.loc[
    combined_benchmarks['SQUAD'].isin(
        ['TRAINING', 'NATIONAL']
    )
].copy()

combined_benchmarks['BENCHMARK'] = pd.to_numeric(
    combined_benchmarks['BENCHMARK'],
    errors='coerce'
)


# Recalculate tier thresholds independently for each squad.
combined_benchmarks['2%'] = (
    combined_benchmarks['BENCHMARK'] * 0.98
)

combined_benchmarks['3.5%'] = (
    combined_benchmarks['BENCHMARK'] * 0.965
)

combined_benchmarks['5%'] = (
    combined_benchmarks['BENCHMARK'] * 0.95
)

combined_benchmarks['10%'] = (
    combined_benchmarks['BENCHMARK'] * 0.90
)


# ============================================================
# 3. VALIDATE BENCHMARK UNIQUENESS
# ============================================================

duplicate_benchmarks = (
    combined_benchmarks
    .groupby(
        ['EVENT', 'GENDER', 'SQUAD']
    )
    .size()
    .reset_index(name='ROW_COUNT')
)

duplicate_benchmarks = duplicate_benchmarks.loc[
    duplicate_benchmarks['ROW_COUNT'] > 1
]

if not duplicate_benchmarks.empty:

    display(duplicate_benchmarks)

    raise ValueError(
        'Duplicate Training/National benchmark rows detected.'
    )


# ============================================================
# 4. ATTACH BOTH BENCHMARK SETS TO EACH BEST PERFORMANCE
# ============================================================

combined_selection = best_results_base.merge(
    combined_benchmarks[
        [
            'EVENT',
            'GENDER',
            'SQUAD',
            'BENCHMARK',
            '2%',
            '3.5%',
            '5%',
            '10%',
        ]
    ],
    how='inner',
    on=['EVENT', 'GENDER'],
)


# ============================================================
# 5. CALCULATE DELTAS
# ============================================================

combined_selection['RESULT_CONV'] = pd.to_numeric(
    combined_selection['RESULT_CONV'],
    errors='coerce'
)

combined_selection['Delta2'] = (
    combined_selection['RESULT_CONV']
    - combined_selection['2%']
)

combined_selection['Delta3.5'] = (
    combined_selection['RESULT_CONV']
    - combined_selection['3.5%']
)

combined_selection['Delta5'] = (
    combined_selection['RESULT_CONV']
    - combined_selection['5%']
)

combined_selection['Delta10'] = (
    combined_selection['RESULT_CONV']
    - combined_selection['10%']
)

combined_selection['Delta_Benchmark'] = (
    combined_selection['RESULT_CONV']
    - combined_selection['BENCHMARK']
)


# Keep existing PERF_SCALAR definition for parity
# with the current Jumps reports.

combined_selection['PERF_SCALAR'] = (
    combined_selection['Delta5']
    / combined_selection['BENCHMARK']
    * 100
)


# ============================================================
# 6. ASSIGN TIERS INDEPENDENTLY FOR EACH SQUAD
# ============================================================

combined_selection['TIER'] = np.where(

    combined_selection['Delta_Benchmark'] >= 0,
    'Tier 1',

    np.where(
        combined_selection['Delta2'] >= 0,
        'Tier 2',

        np.where(
            combined_selection['Delta3.5'] >= 0,
            'Tier 3',

            np.where(
                combined_selection['Delta5'] >= 0,
                'Tier 4',

                np.where(
                    combined_selection['Delta10'] >= 0,
                    'Tier 5',
                    ' '
                )
            )
        )
    )
)


# ============================================================
# 7. KEEP ATHLETES WITHIN THE TIER WINDOW
# ============================================================

final_tiered_selection = combined_selection.loc[
    combined_selection['TIER'].ne(' ')
].copy()


# ============================================================
# 7A. DERIVE ACTUAL PROGRAMME STATUS
# ============================================================

# Programme membership is determined only by Tier 1:
#
# NATIONAL Tier 1  -> National
# TRAINING Tier 1  -> Training
# otherwise        -> Not Selected
#
# National takes precedence over Training.

programme_group_cols = [
    'NAME',
    'GENDER',
    'EVENT'
]

final_tiered_selection['_IS_NATIONAL_MEMBER'] = (
    final_tiered_selection['SQUAD'].eq('NATIONAL')
    & final_tiered_selection['TIER'].eq('Tier 1')
)

final_tiered_selection['_IS_TRAINING_MEMBER'] = (
    final_tiered_selection['SQUAD'].eq('TRAINING')
    & final_tiered_selection['TIER'].eq('Tier 1')
)

membership_flags = (
    final_tiered_selection
    .groupby(programme_group_cols)[
        [
            '_IS_NATIONAL_MEMBER',
            '_IS_TRAINING_MEMBER'
        ]
    ]
    .transform('max')
)

final_tiered_selection['PROGRAMME_STATUS'] = np.select(
    [
        membership_flags['_IS_NATIONAL_MEMBER'],
        membership_flags['_IS_TRAINING_MEMBER'],
    ],
    [
        'National',
        'Training',
    ],
    default='Not Selected'
)

final_tiered_selection = final_tiered_selection.drop(
    columns=[
        '_IS_NATIONAL_MEMBER',
        '_IS_TRAINING_MEMBER'
    ]
)


# ============================================================
# 8. SORT
# ============================================================

squad_order = pd.Categorical(
    final_tiered_selection['SQUAD'],
    categories=[
        'NATIONAL',
        'TRAINING',
    ],
    ordered=True,
)

# Apply deterministic sort
final_tiered_selection = (
    final_tiered_selection
    .assign(_SQUAD_ORDER=squad_order)
    .sort_values(
        [
            '_SQUAD_ORDER',
            'EVENT',
            'GENDER',
            'PERF_SCALAR',
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
    )
    .drop(columns='_SQUAD_ORDER')
    .reset_index(drop=True)
)


# Rename SQUAD in the final report for clarity.
# TRAINING / NATIONAL refers to the benchmark being evaluated,
# not actual programme membership.

final_tiered_selection = (
    final_tiered_selection
    .rename(
        columns={
            'SQUAD': 'BENCHMARK_LEVEL'
        }
    )
)

# ============================================================
# 9. SUMMARY
# ============================================================

print(
    'Combined Training + National selections:',
    len(final_tiered_selection)
)

print('\nSelection count by squad:')
print(
    final_tiered_selection['BENCHMARK_LEVEL']
    .value_counts()
)

print('\nSelection count by squad and tier:')
print(
    final_tiered_selection
    .groupby(
    ['BENCHMARK_LEVEL', 'TIER']
    )
    .size()
)


display(final_tiered_selection)

Combined Training + National selections: 371

Selection count by squad:
BENCHMARK_LEVEL
TRAINING    274
NATIONAL     97
Name: count, dtype: int64

Selection count by squad and tier:
BENCHMARK_LEVEL  TIER  
NATIONAL         Tier 1     10
                 Tier 2      5
                 Tier 3      9
                 Tier 4      5
                 Tier 5     68
TRAINING         Tier 1     35
                 Tier 2     19
                 Tier 3     24
                 Tier 4     24
                 Tier 5    172
dtype: int64


,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER,PROGRAMME_STATUS
0,Goh Yen Young Amelia,1.69m,Singapore,17.0,4,Open,High Jump,0.0,,Y339G08,...,1.5675,1.485,0.073,0.09775,0.1225,0.205,0.04,7.424242,Tier 1,National
1,Chew Yiu Tse Jade,1.67m,National University of Singapo,0.0,1,Open,High Jump,0.0,,,...,1.5675,1.485,0.053,0.07775,0.1025,0.185,0.02,6.212121,Tier 1,National
2,Sng Suat Li Michelle,1.65m,Singapore,38.0,5,Open,High Jump,0.0,,M825I87,...,1.5675,1.485,0.033,0.05775,0.0825,0.165,0.00,5.000000,Tier 1,National
3,Tan Kylie Chuan Ying,1.62m,ActiveSG Athletics Club,14.0,1,U15,High Jump,0.0,,,...,1.5675,1.485,0.003,0.02775,0.0525,0.135,-0.03,3.181818,Tier 2,Training
4,Ho Sher Zhen,1.61m,Chij St. Nicholas Girls',16,1,Intermediate,High Jump,0.0,,S552J10,...,1.5675,1.485,-0.007,0.01775,0.0425,0.125,-0.04,2.575758,Tier 3,Training
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366,Wilson Shen Kai Tan,13.09,รร.กฬ.สิงคโปร์,,4.0,U16,Triple Jump,,,,...,13.5850,12.870,-0.924,-0.70950,-0.4950,0.220,-1.21,-3.461538,Tier 5,Not Selected
367,Toh Jayden,13.06m,VICTORIA SCHOOL,15,1,U18,Triple Jump,0.0,,J320J11,...,13.5850,12.870,-0.954,-0.73950,-0.5250,0.190,-1.24,-3.671329,Tier 5,Not Selected
368,Kwan Matthias,13.05m,Singapore University of Social,0.0,3,Open,Triple Jump,0.0,,,...,13.5850,12.870,-0.964,-0.74950,-0.5350,0.180,-1.25,-3.741259,Tier 5,Not Selected
369,Chang I Teng,13.04,DHS,,5.0,A,Triple Jump,,,,...,13.5850,12.870,-0.974,-0.75950,-0.5450,0.170,-1.26,-3.811189,Tier 5,Not Selected


In [727]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

final_tiered_selection.to_csv('final_tiered_selection_30aug2026.csv', encoding='utf-8')

In [728]:
# ============================================================
# CELL 61 — PROGRAMME VALIDATION SETTINGS + BENCHMARKS
# ============================================================

import re
import datetime
import numpy as np
import pandas as pd


# ============================================================
# 1. PROGRAMME VALIDATION DATES
# ============================================================

# Manually set these dates for each monthly validation run.
#
# Example:
#   Lookback start     = 01 Jan 2025
#   Previous snapshot  = 31 Jul 2026
#   Current snapshot   = 31 Aug 2026
#
# The two snapshots are cumulative:
#
#   Previous snapshot:
#       PROGRAMME_LOOKBACK_START → PREVIOUS_SNAPSHOT_DATE
#
#   Current snapshot:
#       PROGRAMME_LOOKBACK_START → CURRENT_SNAPSHOT_DATE
# ============================================================

PROGRAMME_LOOKBACK_START = datetime.date(2025, 1, 1)

PREVIOUS_SNAPSHOT_DATE = datetime.date(2026, 1, 1)

CURRENT_SNAPSHOT_DATE = datetime.date(2026, 8, 20)


# ============================================================
# 2. VALIDATE SNAPSHOT DATES
# ============================================================

if PREVIOUS_SNAPSHOT_DATE < PROGRAMME_LOOKBACK_START:
    raise ValueError(
        'Previous snapshot cannot be before the programme lookback start.'
    )

if CURRENT_SNAPSHOT_DATE <= PREVIOUS_SNAPSHOT_DATE:
    raise ValueError(
        'Current snapshot must be later than previous snapshot.'
    )


# ============================================================
# 3. PREPARE PROGRAMME SOURCE DATA
# ============================================================

# df_select has already been:
# - Singapore-filtered
# - name-standardised
# - result-converted
#
# WIND is retained as informational data only.
# It does not affect Jumps Programme eligibility.

programme_source = df_select.copy()


# Keep the original DATE column unchanged.
# Create a separate calendar-date version for Programme Changes.
programme_source['DATE_CALENDAR'] = (
    pd.to_datetime(
        programme_source['DATE'],
        errors='coerce'
    )
    .dt.date
)


# Ensure result values are numeric.
programme_source['RESULT_CONV'] = pd.to_numeric(
    programme_source['RESULT_CONV'],
    errors='coerce'
)


# ============================================================
# 4. CHECK THAT THE UPSTREAM NOTEBOOK DATE WINDOW IS WIDE ENOUGH
# ============================================================

# Cell 10 is the active calendar-date filter earlier in the notebook.
#
# Its source window MUST cover:
#
#   PROGRAMME_LOOKBACK_START
#       through at least
#   CURRENT_SNAPSHOT_DATE
#
# Otherwise results may already have been removed before
# this Programme Changes validation section runs.
#
# Example:
#
#   Cell 10 source:
#       01 Jan 2025 → 31 Dec 2026
#
#   Programme validation:
#       Previous = 31 Jul 2026
#       Current  = 31 Aug 2026
#
# is valid.

upstream_start = pd.Timestamp(start_date).date()
upstream_end = pd.Timestamp(end_date).date()


if upstream_start > PROGRAMME_LOOKBACK_START:
    raise ValueError(
        f'Upstream notebook date window starts at {upstream_start}, '
        f'but Programme Changes requires '
        f'{PROGRAMME_LOOKBACK_START} or earlier.'
    )


if upstream_end < CURRENT_SNAPSHOT_DATE:
    raise ValueError(
        f'Upstream notebook date window ends at {upstream_end}, '
        f'but Programme Changes requires '
        f'{CURRENT_SNAPSHOT_DATE} or later.'
    )


# ============================================================
# 5. RESTRICT TO JUMPS PROGRAMME EVENTS
# ============================================================

# Pole Vault remains intentionally excluded.

PROGRAMME_EVENTS = [
    'Long Jump',
    'Triple Jump',
    'High Jump',
]


programme_source = programme_source.loc[
    programme_source['EVENT'].isin(PROGRAMME_EVENTS)
].copy()


# ============================================================
# 6. SPEXPOTENTIAL / SPEXSCHOLARSHIP EXCLUSIONS
# ============================================================

# Reuse the exact same 12-athlete exclusion list
# already applied before final_tiered_selection.

PROGRAMME_EXCLUDED_NAMES = (
    JUMPS_SPEX_EXCLUDED_NAMES.copy()
)

def programme_name_signature(value):
    """
    Create a narrow order-insensitive name signature.

    This is used only as a fallback for the explicit
    Programme exclusion list.
    """

    tokens = re.findall(
        r'[a-z0-9]+',
        str(value).casefold()
    )

    return '|'.join(sorted(tokens))


def canonicalise_programme_exclusion_name(value):
    """
    Pass Programme exclusion names through the same
    name-standardisation mapping used elsewhere in the notebook.
    """

    try:

        key = _octc_name_match_key(value)

        mapped = name_map.get(key)
        
        if mapped is None:
        
            mapped = token_signature_map.get(
                name_token_signature(value),
                value
            )
            
        return clean_replacement_name(
            mapped
        ).title()

    except Exception:

        return str(
            value
        ).strip().title()


# Canonicalised versions of the explicit exclusion list.
canonical_excluded_names = [

    canonicalise_programme_exclusion_name(name)

    for name in PROGRAMME_EXCLUDED_NAMES
]


# Build signatures for both original and canonical forms.
excluded_signatures = {

    programme_name_signature(name)

    for name in (
        PROGRAMME_EXCLUDED_NAMES
        + canonical_excluded_names
    )

    if programme_name_signature(name)
}


# Create athlete signature in the Programme source.
programme_source['NAME_SIGNATURE'] = (

    programme_source['NAME']
    .apply(programme_name_signature)
)


# Remove excluded athletes.
programme_source = programme_source.loc[

    ~programme_source[
        'NAME_SIGNATURE'
    ].isin(
        excluded_signatures
    )

].copy()


# ============================================================
# 7. CREATE EVENT-SPECIFIC PROGRAMME IDENTITY
# ============================================================

# Programme status is deliberately event-specific:
#
#   Athlete + Gender + Event
#
# Therefore the same athlete can legitimately have:
#
#   Long Jump   → Training
#   Triple Jump → National
#
# as two separate Programme statuses.

try:

    athlete_key = (

        programme_source['NAME']
        .apply(_octc_name_match_key)

    )

except Exception:

    athlete_key = (

        programme_source['NAME']
        .fillna('')
        .astype(str)
        .str.strip()
        .str.casefold()

    )


programme_source['PROGRAMME_KEY'] = (

    athlete_key
    .fillna('')
    .astype(str)
    .str.strip()

    + '|'

    + programme_source['GENDER']
      .fillna('')
      .astype(str)
      .str.strip()
      .str.casefold()

    + '|'

    + programme_source['EVENT']
      .fillna('')
      .astype(str)
      .str.strip()
      .str.casefold()

)


# ============================================================
# 8. LOAD BOTH TRAINING AND NATIONAL BENCHMARKS
# ============================================================

# Reload both benchmark sets independently.
#
# This means the Programme Changes validation does NOT depend
# on whether the earlier standalone notebook section was run
# as TRAINING or NATIONAL.

programme_benchmarks = pd.read_csv(
    'Benchmarks.csv'
)


programme_benchmarks.columns = (

    programme_benchmarks.columns
    .astype(str)
    .str.replace(
        '\ufeff',
        '',
        regex=False
    )
    .str.strip()

)


# Clean only textual benchmark columns.
for col in [
    'SQUAD',
    'EVENT',
    'GENDER',
]:

    programme_benchmarks[col] = (

        programme_benchmarks[col]
        .fillna('')
        .astype(str)
        .str.replace(
            '\xa0',
            ' ',
            regex=False
        )
        .str.strip()

    )


programme_benchmarks['SQUAD'] = (

    programme_benchmarks['SQUAD']
    .str.upper()

)


programme_benchmarks['BENCHMARK'] = pd.to_numeric(

    programme_benchmarks['BENCHMARK'],

    errors='coerce'

)


# ============================================================
# 9. CREATE TRAINING / NATIONAL BENCHMARK LOOKUP
# ============================================================

benchmark_lookup = (

    programme_benchmarks.loc[

        programme_benchmarks['SQUAD'].isin(
            [
                'TRAINING',
                'NATIONAL',
            ]
        )

        &

        programme_benchmarks['EVENT'].isin(
            PROGRAMME_EVENTS
        ),

        [
            'EVENT',
            'GENDER',
            'SQUAD',
            'BENCHMARK',
        ],

    ]

    .pivot_table(

        index=[
            'EVENT',
            'GENDER',
        ],

        columns='SQUAD',

        values='BENCHMARK',

        aggfunc='last',

    )

    .reset_index()

    .rename(
        columns={
            'TRAINING':
                'TRAINING_BENCHMARK',

            'NATIONAL':
                'NATIONAL_BENCHMARK',
        }
    )

)


# ============================================================
# 10. VALIDATE BENCHMARK STRUCTURE
# ============================================================

required_benchmark_cols = {

    'TRAINING_BENCHMARK',
    'NATIONAL_BENCHMARK',

}


missing_benchmark_cols = (

    required_benchmark_cols
    .difference(
        benchmark_lookup.columns
    )

)


if missing_benchmark_cols:

    raise ValueError(

        f'Missing benchmark columns: '
        f'{sorted(missing_benchmark_cols)}'

    )


# Every Programme event/gender combination must have:
#
# National benchmark > Training benchmark
#
# This is required because National takes precedence over Training.

invalid_benchmarks = benchmark_lookup.loc[

    benchmark_lookup[
        'TRAINING_BENCHMARK'
    ].isna()

    |

    benchmark_lookup[
        'NATIONAL_BENCHMARK'
    ].isna()

    |

    (
        benchmark_lookup[
            'NATIONAL_BENCHMARK'
        ]

        <=

        benchmark_lookup[
            'TRAINING_BENCHMARK'
        ]
    )

]


if not invalid_benchmarks.empty:

    print(
        'Invalid benchmark rows:'
    )

    display(
        invalid_benchmarks
    )

    raise ValueError(

        'National benchmark must be strictly higher '
        'than Training benchmark for every event/gender.'

    )


# ============================================================
# 11. VALIDATE THAT ALL EXPECTED EVENT / GENDER BENCHMARKS EXIST
# ============================================================

expected_benchmark_pairs = {

    ('Long Jump', 'Male'),
    ('Long Jump', 'Female'),

    ('Triple Jump', 'Male'),
    ('Triple Jump', 'Female'),

    ('High Jump', 'Male'),
    ('High Jump', 'Female'),

}


actual_benchmark_pairs = set(

    benchmark_lookup[
        [
            'EVENT',
            'GENDER',
        ]
    ]

    .itertuples(
        index=False,
        name=None
    )

)


missing_pairs = (

    expected_benchmark_pairs
    - actual_benchmark_pairs

)


if missing_pairs:

    raise ValueError(

        'Missing Programme benchmark combinations: '
        f'{sorted(missing_pairs)}'

    )


# ============================================================
# 12. DISPLAY VALIDATION SETTINGS
# ============================================================

print(
    'Programme source rows:',
    len(programme_source)
)

print(
    'Upstream notebook source window:',
    upstream_start,
    '→',
    upstream_end
)

print(
    'Programme lookback:',
    PROGRAMME_LOOKBACK_START
)

print(
    'Previous snapshot:',
    PREVIOUS_SNAPSHOT_DATE
)

print(
    'Current snapshot:',
    CURRENT_SNAPSHOT_DATE
)

print(
    'Excluded Programme athletes:',
    len(PROGRAMME_EXCLUDED_NAMES)
)

print(
    'Programme benchmark combinations:',
    len(benchmark_lookup)
)


display(
    benchmark_lookup.sort_values(
        [
            'EVENT',
            'GENDER',
        ]
    )
)

Programme source rows: 4144
Upstream notebook source window: 2025-01-01 → 2026-08-30
Programme lookback: 2025-01-01
Previous snapshot: 2026-01-01
Current snapshot: 2026-08-20
Excluded Programme athletes: 12
Programme benchmark combinations: 6


SQUAD,EVENT,GENDER,NATIONAL_BENCHMARK,TRAINING_BENCHMARK
0,High Jump,Female,1.65,1.57
1,High Jump,Male,2.07,1.95
2,Long Jump,Female,5.70,5.30
3,Long Jump,Male,7.20,6.80
4,Triple Jump,Female,12.00,11.30
5,Triple Jump,Male,15.20,14.30


In [447]:
# ============================================================
# CELL 62 — BUILD ONE CUMULATIVE PROGRAMME SNAPSHOT
# ============================================================
def build_programme_snapshot_validation(source_df, snapshot_date):
    period = source_df.loc[
        source_df['DATE_CALENDAR'].notna()
        & (source_df['DATE_CALENDAR'] >= PROGRAMME_LOOKBACK_START)
        & (source_df['DATE_CALENDAR'] <= snapshot_date)
    ].copy()

    # Any valid numeric jump performance can determine
    # Programme membership. Wind does not affect eligibility.
    period = period.loc[
        period['RESULT_CONV'].notna()
    ].copy()
    # Attach BOTH squad benchmarks to each performance.
    period = period.merge(
        benchmark_lookup,
        how='left',
        on=['EVENT', 'GENDER'],
    )
    period = period.loc[
        period['TRAINING_BENCHMARK'].notna()
        & period['NATIONAL_BENCHMARK'].notna()
    ].copy()

    # All three jumps are higher-is-better.
    # Choose the best mark per canonical athlete + gender + event. 
    # DATE is only a deterministic tie-breaker.
    
    period['_DATE_SORT'] = pd.to_datetime(period['DATE'], errors='coerce')
    best = (
        period
        .sort_values(
            ['PROGRAMME_KEY', 'RESULT_CONV', '_DATE_SORT'],
            ascending=[True, False, False],
        )
        .drop_duplicates('PROGRAMME_KEY', keep='first')
        .drop(columns=['_DATE_SORT'])
        .reset_index(drop=True)
    )

    best['PROGRAMME_STATUS'] = np.select(
        [
            best['RESULT_CONV'] >= best['NATIONAL_BENCHMARK'],
            best['RESULT_CONV'] >= best['TRAINING_BENCHMARK'],
        ],
        ['National', 'Training'],
        default='Not Selected',
    )

    # Programme snapshot contains members only; below-Training athletes are absent.
    snapshot = best.loc[
        best['PROGRAMME_STATUS'].isin(['Training', 'National'])
    ].copy()

    snapshot['PROGRAMME_BENCHMARK'] = np.where(
        snapshot['PROGRAMME_STATUS'].eq('National'),
        snapshot['NATIONAL_BENCHMARK'],
        snapshot['TRAINING_BENCHMARK'],
    )

    return snapshot.reset_index(drop=True)

previous_programme_snapshot = build_programme_snapshot_validation(
    programme_source,
    PREVIOUS_SNAPSHOT_DATE,
)
current_programme_snapshot = build_programme_snapshot_validation(
    programme_source,
    CURRENT_SNAPSHOT_DATE,
)

print('Previous programme members:', len(previous_programme_snapshot))
print('Current programme members:', len(current_programme_snapshot))
print('\nPrevious status counts:')
print(previous_programme_snapshot['PROGRAMME_STATUS'].value_counts())
print('\nCurrent status counts:')
print(current_programme_snapshot['PROGRAMME_STATUS'].value_counts())


Previous programme members: 28
Current programme members: 39

Previous status counts:
PROGRAMME_STATUS
Training    19
National     9
Name: count, dtype: int64

Current status counts:
PROGRAMME_STATUS
Training    28
National    11
Name: count, dtype: int64


In [448]:
# ============================================================
# CELL 65 — COMPARE SNAPSHOTS + INDEPENDENT VERIFICATION
# ============================================================
def compare_programme_snapshots_validation(previous, current):
    previous_by_key = {
        row['PROGRAMME_KEY']: row
        for _, row in previous.iterrows()
    }

    rows = []
    for _, current_row in current.iterrows():
        key = current_row['PROGRAMME_KEY']
        previous_row = previous_by_key.get(key)

        previous_status = (
            'Not Selected'
            if previous_row is None
            else previous_row['PROGRAMME_STATUS']
        )
        current_status = current_row['PROGRAMME_STATUS']

        if previous_row is None:
            change = 'NEW ENTRY'
        elif previous_status == 'Training' and current_status == 'National':
            change = 'UPGRADE'
        else:
            continue

        previous_result = '' if previous_row is None else previous_row.get('RESULT', '')
        previous_wind = '' if previous_row is None else previous_row.get('WIND', '')
        previous_date = '' if previous_row is None else previous_row.get('DATE', '')
        previous_comp = '' if previous_row is None else previous_row.get('COMPETITION', '')
        previous_benchmark = '' if previous_row is None else previous_row.get('PROGRAMME_BENCHMARK', '')

        unique_id = str(current_row.get('UNIQUE_ID', '') or '').strip()
        if unique_id == '' and previous_row is not None:
            unique_id = str(previous_row.get('UNIQUE_ID', '') or '').strip()

        rows.append({
            'CHANGE': change,
            'NAME': current_row.get('NAME', ''),
            'GENDER': current_row.get('GENDER', ''),
            'EVENT': current_row.get('EVENT', ''),
            'PREVIOUS_STATUS': previous_status,
            'CURRENT_STATUS': current_status,
            'PREVIOUS_RESULT': previous_result,
            'CURRENT_RESULT': current_row.get('RESULT', ''),
            'PREVIOUS_WIND': previous_wind,
            'CURRENT_WIND': current_row.get('WIND', ''),
            'PREVIOUS_BENCHMARK': previous_benchmark,
            'CURRENT_BENCHMARK': current_row.get('PROGRAMME_BENCHMARK', ''),
            'PREVIOUS_RESULT_DATE': '' if previous_row is None else pd.to_datetime(previous_date, errors='coerce').strftime('%Y-%m-%d'),
            'CURRENT_RESULT_DATE': pd.to_datetime(current_row.get('DATE', ''), errors='coerce').strftime('%Y-%m-%d'),
            'PREVIOUS_COMPETITION': previous_comp,
            'CURRENT_COMPETITION': current_row.get('COMPETITION', ''),
            'UNIQUE_ID': unique_id,
            'PROGRAMME_KEY': key,
        })

    result = pd.DataFrame(rows)
    if not result.empty:
        change_order = pd.Categorical(
            result['CHANGE'], categories=['UPGRADE', 'NEW ENTRY'], ordered=True
        )
        result = (
            result.assign(_CHANGE_ORDER=change_order)
            .sort_values(['_CHANGE_ORDER', 'EVENT', 'GENDER', 'NAME'])
            .drop(columns='_CHANGE_ORDER')
            .reset_index(drop=True)
        )
    return result

programme_changes_validation = compare_programme_snapshots_validation(
    previous_programme_snapshot,
    current_programme_snapshot,
)

# Independent set-based checks.
previous_keys = set(previous_programme_snapshot['PROGRAMME_KEY'])
current_keys = set(current_programme_snapshot['PROGRAMME_KEY'])
expected_new_entry_keys = current_keys - previous_keys

previous_training_keys = set(
    previous_programme_snapshot.loc[
        previous_programme_snapshot['PROGRAMME_STATUS'].eq('Training'),
        'PROGRAMME_KEY',
    ]
)
current_national_keys = set(
    current_programme_snapshot.loc[
        current_programme_snapshot['PROGRAMME_STATUS'].eq('National'),
        'PROGRAMME_KEY',
    ]
)
expected_upgrade_keys = previous_training_keys & current_national_keys

actual_new_entry_keys = set(
    programme_changes_validation.loc[
        programme_changes_validation['CHANGE'].eq('NEW ENTRY'),
        'PROGRAMME_KEY',
    ] if not programme_changes_validation.empty else []
)
actual_upgrade_keys = set(
    programme_changes_validation.loc[
        programme_changes_validation['CHANGE'].eq('UPGRADE'),
        'PROGRAMME_KEY',
    ] if not programme_changes_validation.empty else []
)

print('Programme changes:', len(programme_changes_validation))
print('Expected NEW ENTRY:', len(expected_new_entry_keys), '| Actual:', len(actual_new_entry_keys))
print('Expected UPGRADE:', len(expected_upgrade_keys), '| Actual:', len(actual_upgrade_keys))
print('NEW ENTRY verification:', expected_new_entry_keys == actual_new_entry_keys)
print('UPGRADE verification:', expected_upgrade_keys == actual_upgrade_keys)

assert expected_new_entry_keys == actual_new_entry_keys
assert expected_upgrade_keys == actual_upgrade_keys

programme_changes_validation

# Optional export for direct comparison with Streamlit.
programme_changes_validation.drop(columns=['PROGRAMME_KEY'], errors='ignore').to_csv(
    'programme_changes_validation.csv',
    index=False,
    encoding='utf-8-sig',
)


Programme changes: 12
Expected NEW ENTRY: 11 | Actual: 11
Expected UPGRADE: 1 | Actual: 1
NEW ENTRY verification: True
UPGRADE verification: True


In [449]:
# ============================================================
# MANUAL PROGRAMME CHANGES VERIFICATION TABLE
# ============================================================

previous_lookup = (
    previous_programme_snapshot
    .set_index('PROGRAMME_KEY')
)

current_lookup = (
    current_programme_snapshot
    .set_index('PROGRAMME_KEY')
)

verification_rows = []

for _, change_row in programme_changes_validation.iterrows():

    key = change_row['PROGRAMME_KEY']

    previous_row = (
        previous_lookup.loc[key]
        if key in previous_lookup.index
        else None
    )

    current_row = (
        current_lookup.loc[key]
        if key in current_lookup.index
        else None
    )

    previous_status = (
        'Not Selected'
        if previous_row is None
        else previous_row['PROGRAMME_STATUS']
    )

    current_status = (
        ''
        if current_row is None
        else current_row['PROGRAMME_STATUS']
    )

    previous_result = (
        ''
        if previous_row is None
        else previous_row['RESULT_CONV']
    )

    current_result = (
        ''
        if current_row is None
        else current_row['RESULT_CONV']
    )

    training_benchmark = (
        ''
        if current_row is None
        else current_row['TRAINING_BENCHMARK']
    )

    national_benchmark = (
        ''
        if current_row is None
        else current_row['NATIONAL_BENCHMARK']
    )

    # --------------------------------------------------------
    # Independently derive expected change
    # --------------------------------------------------------

    if (
        previous_status == 'Not Selected'
        and current_status in ['Training', 'National']
    ):
        expected_change = 'NEW ENTRY'

    elif (
        previous_status == 'Training'
        and current_status == 'National'
    ):
        expected_change = 'UPGRADE'

    else:
        expected_change = ''

    actual_change = change_row['CHANGE']

    verification_rows.append({
        'NAME': change_row['NAME'],
        'GENDER': change_row['GENDER'],
        'EVENT': change_row['EVENT'],

        'PREVIOUS_BEST': previous_result,
        'PREVIOUS_STATUS': previous_status,

        'CURRENT_BEST': current_result,
        'CURRENT_STATUS': current_status,

        'TRAINING_BENCHMARK': training_benchmark,
        'NATIONAL_BENCHMARK': national_benchmark,

        'EXPECTED_CHANGE': expected_change,
        'ACTUAL_CHANGE': actual_change,

        'PASS': expected_change == actual_change,

        'PREVIOUS_RESULT_DATE':
            change_row['PREVIOUS_RESULT_DATE'],

        'CURRENT_RESULT_DATE':
            change_row['CURRENT_RESULT_DATE'],

        'PREVIOUS_COMPETITION':
            change_row['PREVIOUS_COMPETITION'],

        'CURRENT_COMPETITION':
            change_row['CURRENT_COMPETITION'],

        'PREVIOUS_WIND':
            change_row['PREVIOUS_WIND'],

        'CURRENT_WIND':
            change_row['CURRENT_WIND'],

        'UNIQUE_ID':
            change_row['UNIQUE_ID'],
    })


programme_changes_manual_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# SUMMARY
# ============================================================

print(
    'Programme changes checked:',
    len(programme_changes_manual_verification)
)

print(
    'PASS:',
    programme_changes_manual_verification['PASS'].sum()
)

print(
    'FAIL:',
    (~programme_changes_manual_verification['PASS']).sum()
)


# ============================================================
# DISPLAY
# ============================================================

display(
    programme_changes_manual_verification
)


# Show failures separately, if any
programme_changes_failures = (
    programme_changes_manual_verification.loc[
        ~programme_changes_manual_verification['PASS']
    ]
    .copy()
)

if not programme_changes_failures.empty:

    print('PROGRAMME CHANGE VERIFICATION FAILURES:')

    display(
        programme_changes_failures
    )

else:

    print(
        'All reported Programme Changes pass '
        'the manual verification logic.'
    )


# Optional export
programme_changes_manual_verification.to_csv(
    'programme_changes_manual_verification.csv',
    index=False,
    encoding='utf-8-sig'
)

Programme changes checked: 12
PASS: 12
FAIL: 0


,NAME,GENDER,EVENT,PREVIOUS_BEST,PREVIOUS_STATUS,CURRENT_BEST,CURRENT_STATUS,TRAINING_BENCHMARK,NATIONAL_BENCHMARK,EXPECTED_CHANGE,ACTUAL_CHANGE,PASS,PREVIOUS_RESULT_DATE,CURRENT_RESULT_DATE,PREVIOUS_COMPETITION,CURRENT_COMPETITION,PREVIOUS_WIND,CURRENT_WIND,UNIQUE_ID
0,Tan Shou Yi Rei,Male,High Jump,2.04,Training,2.09,National,1.95,2.07,UPGRADE,UPGRADE,True,2025-04-13,2026-05-30,National School Games,22nd Asian U20 Athletics Championships 2026,,,
1,Ang Hui Yi,Female,High Jump,,Not Selected,1.60,Training,1.57,1.65,NEW ENTRY,NEW ENTRY,True,,2026-08-02,,Pesta Sukan Athletics 2026,,,
2,Ho Sher Zhen,Female,High Jump,,Not Selected,1.61,Training,1.57,1.65,NEW ENTRY,NEW ENTRY,True,,2026-03-08,,SA All Comers Meet 3,,,S552J10
3,Ho Yuki,Female,High Jump,,Not Selected,1.61,Training,1.57,1.65,NEW ENTRY,NEW ENTRY,True,,2026-02-08,,SA All Comers Meet 2,,,Y646B06
4,Yen Young Amelia Goh,Female,High Jump,,Not Selected,1.60,Training,1.57,1.65,NEW ENTRY,NEW ENTRY,True,,2026-05-29,,22nd Asian U20 Athletics Championships 2026,,,
5,Saw Xiang Yu,Male,High Jump,,Not Selected,1.95,Training,1.95,2.07,NEW ENTRY,NEW ENTRY,True,,2026-08-02,,Pesta Sukan Athletics 2026,,,
6,Lim Chia Ying Kiara,Female,Long Jump,,Not Selected,5.43,Training,5.30,5.70,NEW ENTRY,NEW ENTRY,True,,2026-04-17,,National School Games,,,
7,Tan Shi Jin Angela,Female,Long Jump,,Not Selected,5.53,Training,5.30,5.70,NEW ENTRY,NEW ENTRY,True,,2026-08-02,,Pesta Sukan Athletics 2026,,+0.0,
8,Tan Tse Teng,Female,Long Jump,,Not Selected,5.38,Training,5.30,5.70,NEW ENTRY,NEW ENTRY,True,,2026-02-01,,SA All Comers Meet 1,,1.7,T158D02
9,Lee Jin Yi Gabriel,Male,Long Jump,,Not Selected,6.91,Training,6.80,7.20,NEW ENTRY,NEW ENTRY,True,,2026-01-25,,IVP Track & Field Championships 2026,,+0.0,


All reported Programme Changes pass the manual verification logic.
